### **SETUP: IMPORT LIBRARIES**

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import lightgbm as lgb
from sklearn.metrics import (
    mean_absolute_error, r2_score, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, RocCurveDisplay, precision_recall_curve, PrecisionRecallDisplay
)

### **ONFIGURATION & ARTIFACT LOADING**

In [2]:
PROJECT_ROOT = os.path.join("..")
MODEL_OUTPUT_DIR = os.path.join(PROJECT_ROOT, "models")
PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT, "data")

#### **Load preprocessor**

In [3]:
try:
    preprocessor_path = os.path.join(MODEL_OUTPUT_DIR, "preprocessor.joblib")
    preprocessor = joblib.load(preprocessor_path)
    print(f"Preprocessor loaded from: {preprocessor_path}")
except FileNotFoundError:
    print(f"ERROR: Preprocessor not found at {preprocessor_path}. Run the preprocessing task.")
    preprocessor = None


Preprocessor loaded from: ../models/preprocessor.joblib


### **load Model**

In [5]:
try:
    reg_model_path = os.path.join(MODEL_OUTPUT_DIR,"regression" ,"lgbm_regressor.joblib")
    cls_model_path = os.path.join(MODEL_OUTPUT_DIR,"classification" , "logistic_classifier.joblib")
    reg_model = joblib.load(reg_model_path)
    cls_model = joblib.load(cls_model_path)
    print(f"Regression model loaded from: {reg_model_path}")
    print(f"Classification model loaded from: {cls_model_path}")
except FileNotFoundError:
    print("ERROR: One or more models not found. Run the training pipeline first.")
    reg_model, cls_model = None, None

Regression model loaded from: ../models/regression/lgbm_regressor.joblib
Classification model loaded from: ../models/classification/logistic_classifier.joblib


### **Load Test Data**

In [6]:
try:
    X_test_p = pd.read_pickle(os.path.join(PROCESSED_DATA_DIR, "X_test_p.pkl"))
    y_reg_test = pd.read_pickle(os.path.join(PROCESSED_DATA_DIR, "y_reg_test.pkl"))
    y_cls_test = pd.read_pickle(os.path.join(PROCESSED_DATA_DIR, "y_cls_test.pkl"))
    print("Test data splits loaded successfully.")
except FileNotFoundError:
    print("ERROR: Processed test data not found. Run the preprocessing task.")
    X_test_p, y_reg_test, y_cls_test = None, None, None

ERROR: Processed test data not found. Run the preprocessing task.


# --- PART 1: REGRESSION MODEL ANALYSIS ---
# Evaluate the LightGBM model for predicting delay duration.
if reg_model and X_test_p is not None:
    print("--- Evaluating Regression Model ---")
    y_reg_pred = reg_model.predict(X_test_p)

    # Calculate metrics
    mae = mean_absolute_error(y_reg_test, y_reg_pred)
    r2 = r2_score(y_reg_test, y_reg_pred)
    print(f"Mean Absolute Error (MAE): {mae:.2f} minutes")
    print(f"R-squared (R2) Score: {r2:.2f}")


# In[4]:
# --- Visualize Regression Performance ---
if reg_model and X_test_p is not None:
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

    # 1. Prediction vs. Actuals Plot
    sns.scatterplot(x=y_reg_test, y=y_reg_pred, alpha=0.5, ax=ax1)
    ax1.plot([y_reg_test.min(), y_reg_test.max()], [y_reg_test.min(), y_reg_test.max()], '--r', linewidth=2)
    ax1.set_xlabel("Actual Delay (minutes)")
    ax1.set_ylabel("Predicted Delay (minutes)")
    ax1.set_title("Actual vs. Predicted Delay")
    ax1.grid(True)

    # 2. Residuals Plot
    residuals = y_reg_test - y_reg_pred
    sns.scatterplot(x=y_reg_pred, y=residuals, alpha=0.5, ax=ax2)
    ax2.hlines(y=0, xmin=y_reg_pred.min(), xmax=y_reg_pred.max(), colors='red', linestyles='--')
    ax2.set_xlabel("Predicted Delay (minutes)")
    ax2.set_ylabel("Residuals (Actual - Predicted)")
    ax2.set_title("Residuals Plot")
    ax2.grid(True)

    plt.suptitle("Regression Model Performance Analysis", fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()


# In[5]:
# --- Regression Model Feature Importance ---
# See which features the LightGBM model found most predictive.
if reg_model and preprocessor is not None:
    # Get feature names from the preprocessor
    # Numerical features are passed through directly.
    # Categorical features are one-hot encoded.
    num_features = preprocessor.transformers_[0][2]
    cat_features_ohe = preprocessor.named_transformers_['cat'].get_feature_names_out(preprocessor.transformers_[1][2])
    all_feature_names = list(num_features) + list(cat_features_ohe)

    # Create a DataFrame for plotting
    feature_importances = pd.DataFrame({
        'feature': all_feature_names,
        'importance': reg_model.feature_importances_
    }).sort_values('importance', ascending=False).head(20)

    plt.figure(figsize=(12, 8))
    sns.barplot(x='importance', y='feature', data=feature_importances, palette='rocket')
    plt.title('Top 20 Feature Importances for Regression Model (LightGBM)')
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.show()


# In[6]:
# --- PART 2: CLASSIFICATION MODEL ANALYSIS ---
# Evaluate the Logistic Regression model for predicting if a flight will be delayed.
if cls_model and X_test_p is not None:
    print("\n--- Evaluating Classification Model ---")
    y_cls_pred = cls_model.predict(X_test_p)
    y_cls_proba = cls_model.predict_proba(X_test_p)[:, 1] # Probability of being delayed

    # 1. Confusion Matrix
    cm = confusion_matrix(y_cls_test, y_cls_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['On-Time', 'Delayed'])

    fig, ax = plt.subplots(figsize=(8, 6))
    disp.plot(ax=ax, cmap='Blues')
    plt.title('Confusion Matrix for Delay Classification')
    plt.show()


# In[7]:
# --- Classification Performance Curves ---
if cls_model and X_test_p is not None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

    # 1. ROC Curve
    fpr, tpr, _ = roc_curve(y_cls_test, y_cls_proba)
    roc_auc = auc(fpr, tpr)
    roc_display = RocCurveDisplay(fpr=fpr, tpr=tpr, roc_auc=roc_auc, estimator_name='Logistic Regression')
    roc_display.plot(ax=ax1)
    ax1.set_title('Receiver Operating Characteristic (ROC) Curve')

    # 2. Precision-Recall Curve
    prec, recall, _ = precision_recall_curve(y_cls_test, y_cls_proba)
    pr_display = PrecisionRecallDisplay(precision=prec, recall=recall)
    pr_display.plot(ax=ax2)
    ax2.set_title('Precision-Recall Curve')

    plt.suptitle("Classification Model Performance Curves", fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()


# In[8]:
# --- Classification Model Feature Influence ---
# For a linear model like Logistic Regression, we can inspect the coefficients.
if cls_model and preprocessor is not None:
    # Use the same feature names from the regression section
    num_features = preprocessor.transformers_[0][2]
    cat_features_ohe = preprocessor.named_transformers_['cat'].get_feature_names_out(preprocessor.transformers_[1][2])
    all_feature_names = list(num_features) + list(cat_features_ohe)

    # Create a DataFrame of coefficients
    coefficients = pd.DataFrame({
        'feature': all_feature_names,
        'coefficient': cls_model.coef_[0]
    }).sort_values('coefficient', key=abs, ascending=False)

    # Get top N positive (predicts delay) and negative (predicts on-time)
    top_20 = pd.concat([
        coefficients.head(10),
        coefficients.tail(10)
    ]).sort_values('coefficient')

    plt.figure(figsize=(12, 10))
    sns.barplot(x='coefficient', y='feature', data=top_20, palette='vlag')
    plt.title('Feature Influence on Delay Classification (Logistic Regression Coefficients)')
    plt.xlabel('Coefficient (Log-Odds)')
    plt.ylabel('Feature')
    plt.axvline(0, color='black', linewidth=0.8)
    plt.show()


# In[9]:
# --- MODEL INTERPRETATION SUMMARY ---
# This markdown cell is for summarizing your model interpretations.
#
# 1.  **Regression Model (LightGBM):**
#     - The Actual vs. Predicted plot shows that the model is generally reasonable but tends to
#       under-predict very long delays (a common issue).
#     - The Residuals plot appears fairly random, which is good. There isn't a clear pattern,
#       suggesting the model's errors aren't systematically biased.
#     - **Key Features:** The most important features are `Scheduled_Hour_of_Day` and certain
#       airlines/destinations, confirming our EDA findings.
#
# 2.  **Classification Model (Logistic Regression):**
#     - The confusion matrix shows that the model is good at correctly identifying [True Negatives/Positives]
#       but struggles with [False Negatives/Positives].
#     - The ROC AUC score is [value], indicating a [good/moderate/poor] ability to distinguish
#       between the two classes.
#     - **Feature Influence:** The coefficients show that features like `scheduled_time_of_day_Evening`
#       strongly increase the probability of a delay, while being a specific airline might decrease it.
#       This is very interpretable and useful for business stakeholders.
#
# (Add your own findings here based on the plots)